# Разработка товарных рекомендаций на основе ассоциативных правил

# Описание задачи

Имеется 2 файла с данными - `fastfood_transactions.csv` и `test.csv` для обучения и тестирования моделей соответственно.

Необходимо предсказать поле `expected_item` для данных в файле `test.csv` - наиболее вероятный товар на основе ассоциативных правил.

При помощи этой модели интернет-магазин сможет допродавать больше товаров на основе ассоциативных правил, повышая средний чек, глубину чека и другие важные бизнес-метрики.

Ваш файл должен иметь размерность `11` строк на `2` столбца с полями `item_name` и `expected_item`.

Минимальное значение метрики `Accuracy: 0.6`

In [23]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pyfpgrowth

## Загрузим датастет

In [24]:
df = pd.read_csv('fastfood_transactions.csv')
df.head(10)

,transaction_id,item_name
0,1,Чизбургер
1,1,Сметана
2,1,Коктейль (ваниль)
3,1,Чикенбургер
4,2,Чизбургер
5,2,Наггетсы
6,2,Вода с газом 500 мл
7,2,Коктейль (банан)
8,2,Коктейль (ваниль)
9,2,Сырный соус


## Сгруппируем транзакции по `id` и составим список из списков итемов

In [25]:
transactions = df.groupby('transaction_id')['item_name'].apply(list).tolist()
transactions[:10]

[['Чизбургер', 'Сметана', 'Коктейль (ваниль)', 'Чикенбургер'],
 ['Чизбургер',
  'Наггетсы',
  'Вода с газом 500 мл',
  'Коктейль (банан)',
  'Коктейль (ваниль)',
  'Сырный соус',
  'Вода без газа 500 мл',
  'Картофель фри 200 гр',
  'Кетчуп'],
 ['Картофель фри 300 гр',
  'Чизбургер',
  'Кола',
  'Супер Комбо Обед',
  'Чесночный соус',
  'Вода с газом 500 мл',
  'Коктейль (ваниль)',
  'Сырный соус',
  'Вода без газа 500 мл'],
 ['Цезарь', 'Вода с газом 500 мл', 'Стрипсы'],
 ['Кола',
  'Греческий салат',
  'Чикенбургер',
  'Вода с газом 500 мл',
  'Вода без газа 500 мл',
  'Морковные палочки'],
 ['Кола',
  'Супер Комбо Обед',
  'Ролл-цезарь',
  'Греческий салат',
  'Коктейль (ваниль)',
  'Кетчуп'],
 ['Сметана',
  'Коктейль (ваниль)',
  'Картофель фри 200 гр',
  'Морковные палочки',
  'Кетчуп'],
 ['Кола',
  'Чикенбургер',
  'Сырный соус',
  'Вода без газа 500 мл',
  'Картофель фри 200 гр',
  'Морковные палочки',
  'Кетчуп'],
 ['Цезарь', 'Сметана', 'Вода с газом 500 мл', 'Коктейль (банан)']

In [26]:
# Транзакционный энкодер для Apriori
te = TransactionEncoder()
te_ary = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
df_encoded 
# В итоге получаем One-Hot вектора

,Вода без газа 500 мл,Вода с газом 500 мл,Греческий салат,Картофель фри 200 гр,Картофель фри 300 гр,Кетчуп,Коктейль (банан),Коктейль (ваниль),Кола,Морковные палочки,Наггетсы,Ролл-цезарь,Сметана,Стрипсы,Супер Комбо Обед,Сырный соус,Цезарь,Чесночный соус,Чизбургер,Чикенбургер
0,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,True
1,True,True,False,True,False,True,True,True,False,False,True,False,False,False,False,True,False,False,True,False
2,True,True,False,False,True,False,False,True,True,False,False,False,False,False,True,True,False,True,True,False
3,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False
4,True,True,True,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,True,True,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
19996,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,True
19997,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,True
19998,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,False,False,False,False


In [27]:
# Нахождение частых наборов через Apriori
frequent_itemsets = apriori(
    df=df_encoded,      # датафрейм One-Hot вектора
    min_support=0.05,   # Вероятность, которая будет обрубать ноды и дочерние ноды
    use_colnames=True   # Сохраняем названия столбцов
    )
frequent_itemsets

,support,itemsets
0,0.20710,frozenset({Вода без газа 500 мл})
1,0.28705,frozenset({Вода с газом 500 мл})
2,0.20080,frozenset({Греческий салат})
3,0.20000,frozenset({Картофель фри 200 гр})
4,0.20445,frozenset({Картофель фри 300 гр})
...,...,...
150,0.05115,"frozenset({Цезарь, Чесночный соус})"
151,0.05365,"frozenset({Цезарь, Чизбургер})"
152,0.05155,"frozenset({Чикенбургер, Цезарь})"
153,0.06865,"frozenset({Сметана, Греческий салат, Морковные..."


In [ ]:
# Генерация ассоциативных правил через Apriori
rules_apriori = association_rules(
    df=frequent_itemsets,           # Датафрейм из частых наборов
    metric='confidence',            # метрика
    min_threshold=0.5,              # порог отсечения
    num_itemsets=len(transactions)  # количество элементов во множестве
)
rules_apriori
# antecedents - элементы
# consequents - что следует из элемента
# фактически: if antecedents then consequents

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Греческий салат}),frozenset({Морковные палочки}),0.20080,0.28280,0.11820,0.588645,2.081490,1.0,0.061414,1.743508,0.650119,0.323481,0.426444,0.503304
1,frozenset({Картофель фри 200 гр}),frozenset({Кетчуп}),0.20000,0.28405,0.11945,0.597250,2.102623,1.0,0.062640,1.777654,0.655504,0.327619,0.437461,0.508887
2,frozenset({Картофель фри 300 гр}),frozenset({Сырный соус}),0.20445,0.28380,0.12190,0.596234,2.100894,1.0,0.063877,1.773799,0.658679,0.332742,0.436238,0.512881
3,frozenset({Наггетсы}),frozenset({Коктейль (банан)}),0.19525,0.28145,0.11540,0.591037,2.099972,1.0,0.060447,1.757005,0.650889,0.319402,0.430850,0.500528
4,frozenset({Чизбургер}),frozenset({Коктейль (ваниль)}),0.19855,0.27980,0.11580,0.583228,2.084447,1.0,0.060246,1.728045,0.649144,0.319404,0.421311,0.498548
5,frozenset({Супер Комбо Обед}),frozenset({Кола}),0.20275,0.35740,0.13015,0.641924,1.796093,1.0,0.057687,1.794589,0.555956,0.302674,0.442769,0.503041
6,frozenset({Чикенбургер}),frozenset({Кола}),0.19705,0.35740,0.12355,0.626998,1.754332,1.0,0.053124,1.722780,0.535503,0.286725,0.419543,0.486345
7,frozenset({Сметана}),frozenset({Морковные палочки}),0.31635,0.28280,0.16830,0.532006,1.881208,1.0,0.078836,1.532497,0.685185,0.390623,0.347470,0.563563
8,frozenset({Морковные палочки}),frozenset({Сметана}),0.28280,0.31635,0.16830,0.595120,1.881208,1.0,0.078836,1.688526,0.653133,0.390623,0.407767,0.563563
9,frozenset({Ролл-цезарь}),frozenset({Цезарь}),0.19850,0.28295,0.11910,0.600000,2.120516,1.0,0.062934,1.792625,0.659285,0.328688,0.442159,0.510461


In [40]:
print('Ассоциативные правила (Apriori):')
rules = rules_apriori[['antecedents', 'consequents', 'support', 'confidence']].sort_values(by='support', ascending=False)
rules

Ассоциативные правила (Apriori):


,antecedents,consequents,support,confidence
8,frozenset({Морковные палочки}),frozenset({Сметана}),0.16830,0.595120
7,frozenset({Сметана}),frozenset({Морковные палочки}),0.16830,0.532006
5,frozenset({Супер Комбо Обед}),frozenset({Кола}),0.13015,0.641924
6,frozenset({Чикенбургер}),frozenset({Кола}),0.12355,0.626998
2,frozenset({Картофель фри 300 гр}),frozenset({Сырный соус}),0.12190,0.596234
1,frozenset({Картофель фри 200 гр}),frozenset({Кетчуп}),0.11945,0.597250
9,frozenset({Ролл-цезарь}),frozenset({Цезарь}),0.11910,0.600000
0,frozenset({Греческий салат}),frozenset({Морковные палочки}),0.11820,0.588645
4,frozenset({Чизбургер}),frozenset({Коктейль (ваниль)}),0.11580,0.583228
3,frozenset({Наггетсы}),frozenset({Коктейль (банан)}),0.11540,0.591037


## Функция для предсказания

In [44]:
def find_expected_item(item):
    # Проверяем, есть ли правило, где этот товар является антецедентом
    for idx, row in rules.iterrows():
        if item in row['antecedents']:
            # Если антецедент состоит только из этого товара
            if len(row['antecedents']) == 1:
                return list(row['consequents'])[0]
    
    # Если правило с одним товаром не найдено, ищем правила с несколькими товарами
    # и выбираем правило с наивысшей уверенностью (confidence)
    best_rule = None
    best_confidence = 0
    
    for idx, row in rules.iterrows():
        if item in row['antecedents'] and len(row['antecedents']) > 1:
            if row['confidence'] > best_confidence:
                best_confidence = row['confidence']
                best_rule = row
    
    if best_rule is not None:
        return list(best_rule['consequents'])[0]
    
    return 'Кола'  # Если правило не найдено

In [45]:
test = pd.read_csv('test.csv')
test

,item_name
0,Картофель фри 200 гр
1,Картофель фри 300 гр
2,Чизбургер
3,Чикенбургер
4,Греческий салат
5,Цезарь
6,Ролл-цезарь
7,Супер Комбо Обед
8,Наггетсы
9,Стрипсы


In [46]:
# Применяем функцию к каждому товару
test['expected_item'] = test['item_name'].apply(find_expected_item)
print(test)

               item_name      expected_item
0   Картофель фри 200 гр             Кетчуп
1   Картофель фри 300 гр        Сырный соус
2              Чизбургер  Коктейль (ваниль)
3            Чикенбургер               Кола
4        Греческий салат  Морковные палочки
5                 Цезарь               Кола
6            Ролл-цезарь             Цезарь
7       Супер Комбо Обед               Кола
8               Наггетсы   Коктейль (банан)
9                Стрипсы               Кола
10     Морковные палочки            Сметана


In [48]:
test.to_csv('result.csv', index=False)

## Итоговое значение Accuracy: 0.9090909090909091